# Chest X-ray — Multi-label Classifier (4 classes)

Fine-tunes an ImageNet-pretrained **ResNet18** or **DenseNet121** for **multi-label** chest X-ray classification.

| Class | Meaning |
|-------|---------|
| `Normal` | No findings |
| `Pneumonia` | Pneumonia present |
| `COVID-19` | COVID-19 pattern |
| `Tuberculosis` | TB present |

A single image may have **zero, one, or several** disease labels (non-exclusive).  
Training uses **raw logits + `BCEWithLogitsLoss`** (sigmoid is applied only when computing probabilities / inference — do not use softmax).

**Outputs**
- Best checkpoint → `backend/model_weights/chest_xray/best_model.pth`
- Metrics JSON → `backend/model_weights/chest_xray/metrics_report.json`
- Per-class precision / recall / F1 and **macro-AUC**

**Expected data layout** (folder = positive for that class; images may appear in multiple class folders for multi-label)
```
data/chest_xray/
  train/{Normal,Pneumonia,COVID-19,Tuberculosis}/*
  val/{...}/*      # optional — carved from train if missing
  test/{...}/*     # optional — carved from train if missing
```

Optional CSV (true multi-hot): `data/chest_xray/labels.csv` with columns  
`path,split,Normal,Pneumonia,COVID-19,Tuberculosis` (0/1).

## 1. Imports & configuration

In [ ]:
from __future__ import annotations

import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import models, transforms
from tqdm.auto import tqdm

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "chest_xray"
WEIGHTS_DIR = REPO_ROOT / "backend" / "model_weights" / "chest_xray"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = WEIGHTS_DIR / "best_model.pth"
METRICS_PATH = WEIGHTS_DIR / "metrics_report.json"
LABELS_CSV = DATA_DIR / "labels.csv"

CLASS_NAMES = ["Normal", "Pneumonia", "COVID-19", "Tuberculosis"]
NUM_CLASSES = len(CLASS_NAMES)
ARCH = "resnet18"  # "resnet18" | "densenet121"
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.15
TEST_RATIO = 0.15
NUM_WORKERS = 0
SEED = 42
POS_THRESHOLD = 0.5  # sigmoid threshold for discrete metrics

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Data dir: {DATA_DIR}")
print(f"Checkpoint: {BEST_MODEL_PATH}")
print(f"Architecture: {ARCH} | classes: {CLASS_NAMES}")

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

## 2. Transforms (augmentation)

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(12),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

## 3. Multi-label dataset

Loads either a `labels.csv` (preferred for true multi-hot) or class folders under each split.  
If the same file path appears under multiple class folders, labels are OR-merged into a multi-hot vector.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _empty_label() -> np.ndarray:
    return np.zeros(NUM_CLASSES, dtype=np.float32)


def collect_from_folders(root: Path, split: str) -> list[tuple[Path, np.ndarray]]:
    """Merge class folders into multi-hot rows (path → OR of class bits)."""
    split_dir = root / split
    by_path: dict[str, np.ndarray] = {}
    if not split_dir.exists():
        return []

    for class_idx, name in enumerate(CLASS_NAMES):
        class_dir = split_dir / name
        if not class_dir.is_dir():
            continue
        for p in class_dir.rglob("*"):
            if p.suffix.lower() not in IMAGE_EXTS:
                continue
            key = str(p.resolve())
            if key not in by_path:
                by_path[key] = _empty_label()
            by_path[key][class_idx] = 1.0

            # Mutual exclusivity heuristic: if disease is present, clear Normal
            if name != "Normal" and by_path[key][CLASS_NAMES.index("Normal")] == 1.0:
                by_path[key][CLASS_NAMES.index("Normal")] = 0.0

    return [(Path(k), v) for k, v in sorted(by_path.items())]


def collect_from_csv(csv_path: Path, split: str) -> list[tuple[Path, np.ndarray]]:
    df = pd.read_csv(csv_path)
    required = {"path", "split", *CLASS_NAMES}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"labels.csv missing columns: {sorted(missing)}")

    rows: list[tuple[Path, np.ndarray]] = []
    for _, r in df[df["split"] == split].iterrows():
        p = Path(str(r["path"]))
        if not p.is_absolute():
            p = (DATA_DIR / p).resolve()
        if not p.exists():
            continue
        y = np.array([float(r[c]) for c in CLASS_NAMES], dtype=np.float32)
        rows.append((p, y))
    return rows


class MultiLabelChestXrayDataset(Dataset):
    def __init__(self, samples: list[tuple[Path, np.ndarray]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, torch.from_numpy(label.copy())


def load_split_samples(split: str) -> list[tuple[Path, np.ndarray]]:
    if LABELS_CSV.exists():
        rows = collect_from_csv(LABELS_CSV, split)
        if rows:
            return rows
    return collect_from_folders(DATA_DIR, split)


print("Dataset helpers ready.")

## 4. Bootstrap demo data (optional)

If `data/chest_xray` is empty, generate a small synthetic multi-label set so the pipeline is runnable end-to-end. Replace with a real CXR corpus for meaningful metrics.

In [ ]:
def _synthetic_cxr(seed: int, diseases: list[str], size: int = 256) -> Image.Image:
    rng = random.Random(seed)
    img = Image.new("L", (size, size), color=18 + rng.randint(0, 15))
    draw = ImageDraw.Draw(img)
    draw.ellipse((28, 36, 118, 222), fill=85 + rng.randint(0, 35))
    draw.ellipse((138, 36, 228, 222), fill=85 + rng.randint(0, 35))

    def blotch(color: int, n: int = 6):
        for _ in range(n):
            x0 = rng.randint(35, 190)
            y0 = rng.randint(50, 190)
            draw.ellipse(
                (x0, y0, x0 + rng.randint(18, 48), y0 + rng.randint(18, 48)),
                fill=color,
            )

    if "Pneumonia" in diseases:
        blotch(155, 8)
    if "COVID-19" in diseases:
        blotch(130, 10)
    if "Tuberculosis" in diseases:
        blotch(170, 5)
        for _ in range(4):
            x = rng.randint(40, 200)
            y = rng.randint(40, 100)
            draw.ellipse((x, y, x + 12, y + 12), fill=200)

    img = img.filter(ImageFilter.GaussianBlur(radius=1.1))
    img = ImageEnhance.Contrast(img).enhance(1.15)
    return img.convert("RGB")


def bootstrap_synthetic_dataset(n_per_pattern: int = 24) -> None:
    """Create folder + CSV demo with exclusive and co-occurring labels."""
    patterns = [
        ["Normal"],
        ["Pneumonia"],
        ["COVID-19"],
        ["Tuberculosis"],
        ["Pneumonia", "COVID-19"],
        ["Pneumonia", "Tuberculosis"],
        ["COVID-19", "Tuberculosis"],
    ]
    rows = []
    img_root = DATA_DIR / "images"
    img_root.mkdir(parents=True, exist_ok=True)

    idx = 0
    for pattern in patterns:
        for i in range(n_per_pattern):
            # rough split assignment
            r = (idx % 10) / 10.0
            split = "train" if r < 0.7 else "val" if r < 0.85 else "test"
            fname = f"{'_'.join(pattern).replace('-', '')}_{i:03d}.png"
            rel = Path("images") / fname
            out = DATA_DIR / rel
            _synthetic_cxr(seed=idx + 7, diseases=pattern).save(out)

            # also mirror into class folders for folder-based loaders
            for cls in pattern:
                dest_dir = DATA_DIR / split / cls
                dest_dir.mkdir(parents=True, exist_ok=True)
                link = dest_dir / fname
                if not link.exists():
                    Image.open(out).save(link)

            label_bits = {c: (1 if c in pattern else 0) for c in CLASS_NAMES}
            rows.append({"path": str(rel).replace("\\", "/"), "split": split, **label_bits})
            idx += 1

    pd.DataFrame(rows).to_csv(LABELS_CSV, index=False)
    print(f"Wrote {len(rows)} synthetic samples → {DATA_DIR}")


existing = collect_from_folders(DATA_DIR, "train") or (
    collect_from_csv(LABELS_CSV, "train") if LABELS_CSV.exists() else []
)
if len(existing) < 8:
    print("Few/no training images found — generating synthetic multi-label demo set…")
    bootstrap_synthetic_dataset()
else:
    print(f"Found {len(existing)} train samples — skipping bootstrap.")

## 5. Train / val / test splits

In [ ]:
train_samples = load_split_samples("train")
val_samples = load_split_samples("val")
test_samples = load_split_samples("test")

# Carve val/test from train when missing
if not val_samples or not test_samples:
    pool = list(train_samples)
    if not pool:
        raise RuntimeError(f"No samples under {DATA_DIR}. Add images or re-run bootstrap.")
    rng = random.Random(SEED)
    rng.shuffle(pool)
    n = len(pool)
    n_test = max(1, int(n * TEST_RATIO)) if not test_samples else 0
    n_val = max(1, int(n * VAL_RATIO)) if not val_samples else 0
    cursor = 0
    if not test_samples:
        test_samples = pool[cursor : cursor + n_test]
        cursor += n_test
    if not val_samples:
        val_samples = pool[cursor : cursor + n_val]
        cursor += n_val
    train_samples = pool[cursor:]
    if len(train_samples) < 4:
        # tiny-data fallback: reuse pool for train
        train_samples = pool

train_ds = MultiLabelChestXrayDataset(train_samples, transform=train_transform)
val_ds = MultiLabelChestXrayDataset(val_samples, transform=eval_transform)
test_ds = MultiLabelChestXrayDataset(test_samples, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


def label_prevalence(samples: list[tuple[Path, np.ndarray]]) -> dict[str, float]:
    if not samples:
        return {}
    mat = np.stack([y for _, y in samples], axis=0)
    return {c: float(mat[:, i].mean()) for i, c in enumerate(CLASS_NAMES)}


print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print("Train prevalence:", label_prevalence(train_samples))
print("Val prevalence:  ", label_prevalence(val_samples))
print("Test prevalence: ", label_prevalence(test_samples))

## 6. Model — pretrained backbone + 4-logit head

The classifier outputs **logits** (no sigmoid in `forward`).  
`BCEWithLogitsLoss` applies sigmoid internally for the loss; use `torch.sigmoid` for probabilities at eval / serving time.

In [ ]:
def build_multilabel_model(arch: str = ARCH, num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    arch = arch.lower()
    if arch == "resnet18":
        try:
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            model = models.resnet18(weights=weights)
        except Exception:
            model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == "densenet121":
        try:
            weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
            model = models.densenet121(weights=weights)
        except Exception:
            model = models.densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported arch: {arch}")
    return model


model = build_multilabel_model(ARCH, NUM_CLASSES, pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")
print(model)

## 7. Training helpers

In [ ]:
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss = 0.0
    n = 0
    all_logits = []
    all_targets = []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, targets in tqdm(loader, leave=False):
            images = images.to(device)
            targets = targets.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            if train:
                loss.backward()
                optimizer.step()
            bs = images.size(0)
            total_loss += loss.item() * bs
            n += bs
            all_logits.append(logits.detach().cpu())
            all_targets.append(targets.detach().cpu())

    logits = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))  # sigmoid
    return total_loss / max(n, 1), probs, targets


def multilabel_metrics(probs: np.ndarray, targets: np.ndarray, threshold: float = POS_THRESHOLD) -> dict:
    preds = (probs >= threshold).astype(np.float32)
    per_class = {}
    for i, name in enumerate(CLASS_NAMES):
        y_true = targets[:, i]
        y_pred = preds[:, i]
        y_prob = probs[:, i]
        entry = {
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "support": int(y_true.sum()),
        }
        # AUC needs both classes present
        if len(np.unique(y_true)) > 1:
            entry["auc"] = float(roc_auc_score(y_true, y_prob))
            entry["ap"] = float(average_precision_score(y_true, y_prob))
        else:
            entry["auc"] = float("nan")
            entry["ap"] = float("nan")
        per_class[name] = entry

    f1s = [per_class[c]["f1"] for c in CLASS_NAMES]
    aucs = [per_class[c]["auc"] for c in CLASS_NAMES if not np.isnan(per_class[c]["auc"])]
    return {
        "per_class": per_class,
        "macro_f1": float(np.mean(f1s)),
        "macro_auc": float(np.mean(aucs)) if aucs else float("nan"),
        "subset_accuracy": float((preds == targets).all(axis=1).mean()),
    }


print("Training helpers ready.")

## 8. Train with checkpointing

Best model is selected by **validation macro-AUC** (falls back to macro-F1 if AUC is undefined).

In [ ]:
history = []
best_score = -1.0
best_epoch = -1
t0 = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_probs, train_tgts = run_epoch(model, train_loader, optimizer)
    val_loss, val_probs, val_tgts = run_epoch(model, val_loader, optimizer=None)
    scheduler.step()

    train_m = multilabel_metrics(train_probs, train_tgts)
    val_m = multilabel_metrics(val_probs, val_tgts)
    score = val_m["macro_auc"]
    if np.isnan(score):
        score = val_m["macro_f1"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_macro_f1": train_m["macro_f1"],
        "val_macro_f1": val_m["macro_f1"],
        "val_macro_auc": val_m["macro_auc"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f}  "
        f"val_macro_f1={val_m['macro_f1']:.3f} val_macro_auc={val_m['macro_auc']:.3f}"
    )

    if score > best_score:
        best_score = score
        best_epoch = epoch
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "architecture": ARCH,
            "class_names": CLASS_NAMES,
            "num_classes": NUM_CLASSES,
            "multi_label": True,
            "image_size": IMAGE_SIZE,
            "threshold": POS_THRESHOLD,
            "epoch": epoch,
            "val_macro_auc": val_m["macro_auc"],
            "val_macro_f1": val_m["macro_f1"],
            "val_metrics": val_m,
        }
        torch.save(checkpoint, BEST_MODEL_PATH)
        print(f"  ↳ saved best → {BEST_MODEL_PATH} (score={score:.4f})")

elapsed = time.perf_counter() - t0
print(f"\nTraining done in {elapsed/60:.1f} min | best epoch={best_epoch} score={best_score:.4f}")

## 9. Test evaluation (per-class P/R/F1 + macro-AUC)

In [ ]:
# Load best checkpoint
ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_loss, test_probs, test_tgts = run_epoch(model, test_loader, optimizer=None)
test_metrics = multilabel_metrics(test_probs, test_tgts)
preds = (test_probs >= POS_THRESHOLD).astype(int)

print(f"Test loss: {test_loss:.4f}")
print(f"Macro-F1:  {test_metrics['macro_f1']:.4f}")
print(f"Macro-AUC: {test_metrics['macro_auc']:.4f}")
print(f"Subset accuracy (exact multi-hot match): {test_metrics['subset_accuracy']:.4f}\n")

print(classification_report(
    test_tgts,
    preds,
    target_names=CLASS_NAMES,
    zero_division=0,
))

print("Per-class detail:")
for name, m in test_metrics["per_class"].items():
    print(
        f"  {name:14s}  P={m['precision']:.3f}  R={m['recall']:.3f}  "
        f"F1={m['f1']:.3f}  AUC={m['auc']:.3f}  support={m['support']}"
    )

report = {
    "architecture": ARCH,
    "class_names": CLASS_NAMES,
    "multi_label": True,
    "threshold": POS_THRESHOLD,
    "best_epoch": best_epoch,
    "best_val_score": best_score,
    "test_loss": test_loss,
    "test_metrics": test_metrics,
    "history": history,
    "checkpoint": str(BEST_MODEL_PATH),
}
METRICS_PATH.write_text(json.dumps(report, indent=2))
print(f"\nWrote metrics → {METRICS_PATH}")
print(f"Best model → {BEST_MODEL_PATH}")

## 10. Quick inference smoke test

Matches the backend contract: sigmoid probabilities, threshold positives, return multi-label list.

In [ ]:
@torch.no_grad()
def predict_multilabel(image_path: Path, threshold: float = POS_THRESHOLD) -> list[dict]:
    image = Image.open(image_path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)
    logits = model(tensor)
    probs = torch.sigmoid(logits).cpu().numpy()[0]
    hits = [
        {"label": CLASS_NAMES[i], "confidence": float(probs[i])}
        for i in range(NUM_CLASSES)
        if probs[i] >= threshold
    ]
    if not hits:
        top = int(np.argmax(probs))
        hits = [{"label": CLASS_NAMES[top], "confidence": float(probs[top])}]
    hits.sort(key=lambda x: x["confidence"], reverse=True)
    return hits


sample_path = test_samples[0][0]
print("Sample:", sample_path)
print("True:  ", {CLASS_NAMES[i]: float(test_samples[0][1][i]) for i in range(NUM_CLASSES)})
print("Pred:  ", predict_multilabel(sample_path))